# CORD-19 — Conversione & Sanificazione (JSON/CSV → Parquet)

Questo notebook trasforma il dump grezzo **CORD-19** (`archive/`: `metadata.csv` +
`document_parses/{pdf_json,pmc_json}/`) nei dataset **Parquet** che alimentano le quattro
task del progetto. È l'unione — rivista e commentata — dei vecchi script
`cord19_convert.py`, `build_parquet.py`, `build_silver.py`, `full_sanity.py`.

## Modello dei dati: due layer, tre tabelle

Modello logico **relazionale/normalizzato**, fisico **Parquet partizionato**. Due layer:

- **BRONZE** — estrazione fedele dai file grezzi, con solo un *gate strutturale*: un file
  illeggibile produce zero righe, non un errore.
- **SILVER** — pulizia semantica: dedup, coercizione dei tipi, canonicalizzazione dei paesi,
  flag utili. **Principio**: correggo errori oggettivi e aggiungo flag, **non** prendo
  decisioni di analisi al posto delle task (niente dedup dei titoli, niente stopword, niente
  rimozione delle reference).

Tre tabelle, tutte con chiave `cord_uid`:

| tabella | granularità | sorgente | task |
|---|---|---|---|
| `papers` | 1 riga/paper | `metadata.csv` | 3/4 (titoli) |
| `paragraphs` | 1 riga/paragrafo | `pmc_json` \| `pdf_json` | 1 (word-count) |
| `authors` | 1 riga/(paper, autore) | `pdf_json` | 2 (paesi/istituti) |

Schema completo dei dataset prodotti: vedi `DATA_DICTIONARY.md`.

## Requisiti
Env conda `mapd-covid`. Dipendenza extra non presente nel comando di creazione dell'env:
`country_converter`. Sulla VM (pyvenv del corso) tipicamente:
`pip install "dask[complete]" pyarrow country_converter asyncssh` — `asyncssh` serve solo se
si avvia il cluster multi-nodo via `SSHCluster` (vedi §2).

## 1 · Configurazione (via variabili d'ambiente)

Tutti i path e le opzioni di run vengono da **variabili d'ambiente con default locali**.
Motivo: lo **stesso** file `.ipynb` gira identico su Mac e sulla VM — sulla VM basta
esportare le env var nella shell, **senza mai editare il notebook**. Così un `git pull`
sulla VM non entra mai in conflitto con modifiche locali al file.

| variabile | default | significato |
|---|---|---|
| `CORD19_ARCHIVE` | `.../archive` | cartella del dump CORD-19 (input) |
| `CORD19_DATA` | `.../data` | root degli output Parquet (git-ignored) |
| `DASK_SCHEDULER` | *(vuoto)* | `tcp://host:8786` per collegarsi a uno scheduler **già avviato** |
| `CORD19_HOSTS` | *(vuoto)* | `scheduler,scheduler,worker1,worker2` → avvia un `SSHCluster` multi-nodo da qui |
| `CORD19_SSH_KEY` | *(vuoto)* | path della chiave privata usata da `SSHCluster` (es. la `.pem` di CloudVeneto) |
| `CORD19_WORKERS` | `4` | worker del `LocalCluster` locale (solo default, senza le due sopra) |
| `CORD19_SAMPLE` | `0` | `0` = run completo; `N` = dry-run su N file per sorgente → `data_sample/` |

Le tre opzioni di cluster sono mutuamente esclusive, in ordine di priorità:
`DASK_SCHEDULER` (scheduler esistente) → `CORD19_HOSTS` (SSHCluster) → altrimenti `LocalCluster`.

In [ ]:
import os

# --- input: dump CORD-19 (Kaggle) ---
ARCHIVE = os.environ.get(
    "CORD19_ARCHIVE",
    "/Users/federicosbarbati/Developer/MAPD-Project/archive",
)
META    = os.path.join(ARCHIVE, "metadata.csv")
PDF_DIR = os.path.join(ARCHIVE, "document_parses", "pdf_json")
PMC_DIR = os.path.join(ARCHIVE, "document_parses", "pmc_json")

# --- output: dataset Parquet derivati (git-ignored) ---
DATA_ROOT = os.environ.get(
    "CORD19_DATA",
    "/Users/federicosbarbati/Developer/MAPD-Project/data",
)

# --- opzioni di run ---
DASK_SCHEDULER = os.environ.get("DASK_SCHEDULER")           # "tcp://host:8786": scheduler già avviato
CORD19_HOSTS   = os.environ.get("CORD19_HOSTS")             # "scheduler,scheduler,worker1,...": SSHCluster
CORD19_SSH_KEY = os.environ.get("CORD19_SSH_KEY")           # path chiave privata per SSHCluster (es. la .pem)
N_WORKERS      = int(os.environ.get("CORD19_WORKERS", "4")) # worker del LocalCluster locale
SAMPLE         = int(os.environ.get("CORD19_SAMPLE", "0"))  # 0 = full run; N = dry-run su N file/sorgente

# i dry-run scrivono in una root separata: non sovrascrivono mai i dataset completi
ROOT = DATA_ROOT + ("_sample" if SAMPLE else "")

# partizioni: ~100-150 MB/parte sul run completo, ridotte sul sample
if SAMPLE:
    NPART_PARA, NPART_AUTH, NPART_PAPERS = 8, 4, 9
else:
    NPART_PARA, NPART_AUTH, NPART_PAPERS = 48, 12, 9

if DASK_SCHEDULER:
    _cluster_desc = f"Client({DASK_SCHEDULER})"
elif CORD19_HOSTS:
    _cluster_desc = f"SSHCluster([{CORD19_HOSTS}])"
else:
    _cluster_desc = f"LocalCluster(n_workers={N_WORKERS})"

print("ARCHIVE :", ARCHIVE)
print("ROOT    :", ROOT)
print("cluster :", _cluster_desc)
print("mode    :", f"SAMPLE={SAMPLE}" if SAMPLE else "FULL RUN")

## 2 · Cluster Dask

La cella è **idempotente**: rieseguirla chiude il cluster precedente invece di *orfanarlo*
(un cluster orfano continua a tenere occupata la porta 8787 della dashboard). Tre modalità, in
ordine di priorità:

1. **`DASK_SCHEDULER`** valorizzato → ci si collega a uno scheduler **già avviato**
   (`Client("tcp://host:8786")`). Utile se lo scheduler/worker li avvii a parte (CLI o Docker).
2. **`CORD19_HOSTS`** valorizzato → si avvia un **`SSHCluster`** multi-nodo **da qui** (metodo del
   corso). La lista è `scheduler,scheduler,worker1,worker2,...`: **il primo host è lo scheduler**,
   **tutti** gli host diventano worker (ripeti lo scheduler per usarlo anche come worker). Gli
   host possono essere hostname o IP interni (es. `10.67.22.x`). **Prerequisiti**: SSH
   *passwordless* dallo scheduler verso ogni nodo (una **chiave unica di cluster**, la stessa su
   tutti i nodi), `CORD19_SSH_KEY` che punta alla chiave privata (la `.pem`), e il pacchetto
   `asyncssh` installato. Non serve indicare le porte dei worker: le assegna lo scheduler.
3. altrimenti → **`LocalCluster`** in-process (sviluppo sul Mac).

Dashboard su `localhost:8787`. Dopo l'avvio, l'oggetto `client` stampa quanti **worker/core/memoria**
si sono registrati: è la verifica che il cluster funziona.

In [ ]:
from dask.distributed import Client, LocalCluster

# idempotenza: chiudi ciò che è già attivo prima di ricreare (evita cluster orfani)
try:
    client.close()
    cluster.close()
except NameError:
    pass

if DASK_SCHEDULER:
    # (1) scheduler già avviato altrove -> mi collego e basta
    cluster = None
    client = Client(DASK_SCHEDULER)
elif CORD19_HOSTS:
    # (2) avvio il cluster multi-nodo via SSH da qui (metodo del corso).
    # hosts[0] = scheduler; TUTTI gli host diventano worker (lo scheduler compare
    # due volte per fare anche da worker). Richiede passwordless SSH + asyncssh.
    from dask.distributed import SSHCluster
    hosts = [h.strip() for h in CORD19_HOSTS.split(",") if h.strip()]
    connect = {"known_hosts": None}                     # non verificare la host key (l'IP cambia)
    if CORD19_SSH_KEY:
        connect["client_keys"] = [CORD19_SSH_KEY]       # chiave privata dello scheduler (es. .pem)
    cluster = SSHCluster(
        hosts,
        connect_options=connect,
        scheduler_options={"port": 8786, "dashboard_address": ":8787"},
    )
    client = Client(cluster)
else:
    # (3) default: cluster locale in-process
    cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=2, processes=True)
    client = Client(cluster)

print("dask dashboard:", client.dashboard_link)
client

## 3 · Import

In [ ]:
import re
import json
import time
import shutil
import random
import unicodedata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import dask.bag as db
import dask.dataframe as dd

## 4 · Lettura file e *linkage*

**Gotcha di lettura**: ogni file JSON è **un singolo oggetto JSON multi-riga**
(pretty-printed), *non* JSON-lines → si legge un file intero per volta (`load_json`), **non**
con `read_text().map(json.loads)` (che spezzerebbe per riga e fallirebbe).

Il *linkage* collega ogni file full-text al suo `cord_uid`. `build_linkage` legge le colonne
di path da `metadata.csv` e costruisce `{filename -> [cord_uid, ...]}` (deduplicando: lo stesso
file può comparire in righe duplicate). `build_workitems` produce work-item **auto-contenuti**
`(filename, [cord_uid...], source)` in cui si memorizza **solo il nome del file**: il path
assoluto viene ricostruito nel worker (`resolve_path`). Così il *task graph* di Dask resta
piccolo e **portabile tra i nodi** del cluster — non si trasmette né un grosso dizionario né
path assoluti legati a una singola macchina.

In [ ]:
def load_json(path):
    """Un singolo oggetto JSON pretty-printed per file (NON json-lines)."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_linkage(meta_path=META):
    """{filename -> [cord_uid, ...]} per pdf e pmc, dalle colonne di path di
    metadata.csv. I cord_uid sono deduplicati per file (metadata ha righe duplicate)."""
    df = pd.read_csv(meta_path, dtype=str,
                     usecols=["cord_uid", "pdf_json_files", "pmc_json_files"])
    pdf_map, pmc_map = {}, {}
    for cu, pdfs, pmcs in zip(df.cord_uid, df.pdf_json_files, df.pmc_json_files):
        if isinstance(pdfs, str):
            for p in pdfs.split(";"):
                p = p.strip()
                if p:
                    pdf_map.setdefault(os.path.basename(p), set()).add(cu)
        if isinstance(pmcs, str):
            for p in pmcs.split(";"):
                p = p.strip()
                if p:
                    pmc_map.setdefault(os.path.basename(p), set()).add(cu)
    pdf_map = {k: sorted(v) for k, v in pdf_map.items()}
    pmc_map = {k: sorted(v) for k, v in pmc_map.items()}
    return pdf_map, pmc_map


def build_workitems(pdf_map, pmc_map, pdf_dir=PDF_DIR, pmc_dir=PMC_DIR):
    """Work-item auto-contenuti [(filename, [cord_uid,...], source), ...].
    Si memorizza solo il filename (path ricostruito nel worker via resolve_path) per
    tenere il task graph Dask piccolo e portabile tra i nodi del cluster."""
    pdf_items, pmc_items, unresolved = [], [], 0
    for fn in os.listdir(pdf_dir):
        cus = pdf_map.get(fn)
        if cus is None:
            unresolved += 1
            continue
        pdf_items.append((fn, cus, "pdf"))
    for fn in os.listdir(pmc_dir):
        cus = pmc_map.get(fn)
        if cus is None:
            unresolved += 1
            continue
        pmc_items.append((fn, cus, "pmc"))
    return pdf_items, pmc_items, unresolved


def resolve_path(filename, source):
    """Ricostruisce il path assoluto nel worker da (filename, source)."""
    return os.path.join(PDF_DIR if source == "pdf" else PMC_DIR, filename)

## 5 · Estrazione dei record

Due funzioni mappate sui work-item. Entrambe hanno un **gate strutturale**: se il file non è
parsabile ritornano `[]` (la riga sparisce, niente crash).

- `extract_paragraphs` → una riga per paragrafo non vuoto. **Gotcha di schema**: `body_text`
  è una chiave **top-level**, *non* dentro `metadata` (lo schema nel PDF del progetto è
  fuorviante su questo punto).
- `extract_authors` → una riga per (paper, autore), **solo da `pdf_json`**: in `pmc_json` le
  affiliazioni sono popolate ~0%. Gli autori senza affiliazione vengono comunque tenuti.

Quando un file è referenziato da più `cord_uid`, la riga viene emessa per ciascuno di essi.

In [ ]:
def clean_str(s):
    """Strip di una stringa; valori vuoti/non-stringa diventano None."""
    if not isinstance(s, str):
        return None
    s = s.strip()
    return s if s else None


def extract_paragraphs(item):
    """(filename, [cord_uid...], source) -> lista di righe-paragrafo.
    Gate strutturale: file non parsabile -> []. Paragrafi con testo vuoto scartati."""
    filename, cord_uids, source = item
    try:
        rec = load_json(resolve_path(filename, source))
    except Exception:
        return []
    pid = rec.get("paper_id")
    rows = []
    # NB: body_text è una chiave TOP-LEVEL, non sotto metadata (gotcha di schema).
    for i, para in enumerate(rec.get("body_text", []) or []):
        text = para.get("text")
        if not isinstance(text, str) or not text.strip():
            continue
        section = clean_str(para.get("section"))
        for cu in cord_uids:
            rows.append({"cord_uid": cu, "paper_id": pid, "source": source,
                         "para_idx": i, "section": section, "text": text})
    return rows


def extract_authors(item):
    """(filename, [cord_uid...], source) -> lista di righe-autore. Solo pdf_json
    (in pmc le affiliazioni sono ~0% popolate). Tiene gli autori anche senza affiliazione."""
    filename, cord_uids, source = item
    try:
        rec = load_json(resolve_path(filename, source))
    except Exception:
        return []
    pid = rec.get("paper_id")
    rows = []
    for idx, a in enumerate(rec.get("metadata", {}).get("authors", []) or []):
        aff = a.get("affiliation") or {}
        loc = aff.get("location") or {}
        for cu in cord_uids:
            rows.append({"cord_uid": cu, "paper_id": pid, "author_idx": idx,
                         "institution": clean_str(aff.get("institution")),
                         "country_raw": clean_str(loc.get("country")),
                         "settlement": clean_str(loc.get("settlement"))})
    return rows

## 6 · Schemi (meta di Dask + schemi Arrow espliciti)

Due cose diverse:
- **`*_META`** — dice a `bag.to_dataframe` la forma/tipi del DataFrame *a monte*: Dask è lazy
  e ha bisogno di conoscere le colonne prima di calcolare.
- **`*_SCHEMA`** (Arrow) — passati a `to_parquet` per **fissare** i tipi su disco invece di
  lasciarli inferire partizione per partizione (partizioni diverse potrebbero dedurre tipi
  incompatibili → merge rotto in lettura).

In [ ]:
# meta = tipi delle colonne per bag.to_dataframe
PARA_META = {"cord_uid": "object", "paper_id": "object", "source": "object",
             "para_idx": "int64", "section": "object", "text": "object"}
AUTH_META = {"cord_uid": "object", "paper_id": "object", "author_idx": "int64",
             "institution": "object", "country_raw": "object", "settlement": "object"}

# schemi Arrow espliciti -> Dask fissa i tipi su disco invece di inferirli per partizione
PARA_SCHEMA = pa.schema([("cord_uid", pa.string()), ("paper_id", pa.string()),
                         ("source", pa.string()), ("para_idx", pa.int32()),
                         ("section", pa.string()), ("text", pa.string())])
AUTH_SCHEMA = pa.schema([("cord_uid", pa.string()), ("paper_id", pa.string()),
                         ("author_idx", pa.int32()), ("institution", pa.string()),
                         ("country_raw", pa.string()), ("settlement", pa.string())])

## 7 · Utility di scrittura

`write_pandas_parquet` scrive un DataFrame pandas (piccolo) come cartella di `nparts` file
Parquet, in **pandas puro** — il frame non entra mai in un task graph Dask.
`downcast_large_string` normalizza `large_string → string` così tutti i dataset condividono lo
stesso tipo Arrow. `fresh_dir` garantisce che ogni build parta da una cartella pulita.

In [ ]:
def fresh_dir(path):
    """Cancella e ricrea una cartella: ogni build parte da uno stato pulito."""
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)


def downcast_large_string(table):
    """large_string -> string, per tipi Arrow uniformi tra tutti i dataset."""
    fields = [pa.field(f.name, pa.string() if f.type == pa.large_string() else f.type)
              for f in table.schema]
    return table.cast(pa.schema(fields))


def write_pandas_parquet(df, path, nparts):
    """Scrive un frame pandas (piccolo) come cartella di nparts file Parquet,
    normalizzando large_string -> string. Pandas puro: il frame non entra in un task graph."""
    fresh_dir(path)
    nparts = min(nparts, max(len(df), 1))
    for i, ix in enumerate(np.array_split(np.arange(len(df)), nparts)):
        t = downcast_large_string(pa.Table.from_pandas(df.iloc[ix], preserve_index=False))
        pq.write_table(t, os.path.join(path, f"part.{i}.parquet"), compression="zstd")


def dir_size_mb(path):
    return sum(os.path.getsize(os.path.join(path, f))
               for f in os.listdir(path) if f.endswith(".parquet")) / 1e6


def report(name, path, t0):
    print(f"  [OK] {name:26s} {dir_size_mb(path):8.1f} MB   {time.time()-t0:6.1f}s")

## 8 · Trasformazioni SILVER (definizioni)

Funzioni di pulizia semantica — definite qui, **applicate** nella sezione build (§11).

- `silver_papers` — `cord_uid` **non è unico**: si tiene la riga con più parse
  (`has_pdf + has_pmc`) per paper. `year` = primi 4 caratteri di `publish_time` (formato `YYYY`
  o `YYYY-MM-DD`; un `pd.to_datetime` ingenuo scarterebbe ~metà delle righe). Elimina le colonne
  morte (`mag_id` ~100% null, `arxiv_id` ~99%, `who_covidence_id` ~60%).
- `enrich_papers` — flag per le task 3/4 senza scartare nulla: `title_norm` (per il match),
  `title_dup_count`/`is_title_unique` (titoli duplicati esatti → coseno banale), `title_ok`;
  ripulisce `abstract`.
- `silver_paragraphs` — **prefer-pmc**: tiene tutte le righe `pmc` più le righe `pdf` dei soli
  paper senza parse pmc (il parse PMC è più pulito/completo).
- **Canonicalizzazione paese** — i valori di `country` sono testo sporco. `canonicalize_country`
  risolve i valori **distinti** in codici ISO3 in **tre passate**: (1) `country_converter`
  vettoriale, (2) patch di alias per nomi in lingua/abbreviazioni, (3) retry sul primo token.
  Si convertono solo i valori distinti (poi rimappati sulla Series) per efficienza.
- `norm_institution` — normalizzazione leggera (unicode, spazi, punteggiatura); la
  disambiguazione completa degli istituti è fuori scope.

In [ ]:
DEAD_COLS = ["mag_id", "arxiv_id", "who_covidence_id"]   # ~100% / ~99% / ~60% null


def silver_papers(df):
    """Dedup su cord_uid tenendo la riga più ricca, + year + flag di parse, - colonne morte."""
    df = df.copy()
    # publish_time è "YYYY" o "YYYY-MM-DD": l'anno sono sempre i primi 4 caratteri
    # (un pd.to_datetime ingenuo scarterebbe silenziosamente ~metà delle righe).
    df["year"] = pd.to_numeric(df["publish_time"].str.slice(0, 4),
                               errors="coerce").astype("Int16")
    df["has_pdf"] = df["pdf_json_files"].notna()
    df["has_pmc"] = df["pmc_json_files"].notna()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    # cord_uid non è unico: per paper tieni la riga con più parse (pdf+pmc).
    df["_rank"] = df["has_pdf"].astype(int) + df["has_pmc"].astype(int)
    df = (df.sort_values("_rank", ascending=False)
            .drop_duplicates("cord_uid", keep="first")
            .drop(columns=["_rank"]))
    return df


def enrich_papers(df):
    """Aggiunge flag sui titoli per le task 3/4 (non scarta nulla): title_norm per il match;
    flag di titolo duplicato (titoli identici -> coseno banale); title_ok."""
    df = df.copy()
    norm = (df["title"].fillna("").str.strip().str.lower()
            .str.replace(r"\s+", " ", regex=True))
    df["title_norm"] = norm.where(norm != "", other=None)
    vc = df["title_norm"].value_counts()
    df["title_dup_count"] = df["title_norm"].map(vc).astype("Int32")
    df["is_title_unique"] = df["title_dup_count"].eq(1)
    df["title_ok"] = df["title"].notna() & (df["title"].str.strip().str.len() >= 3)
    if "abstract" in df.columns:
        df["abstract"] = df["abstract"].str.replace(r"\s+", " ", regex=True).str.strip()
    return df

In [ ]:
def silver_paragraphs(paragraphs_ddf):
    """Prefer pmc su pdf per paper: tiene tutte le righe pmc, più le righe pdf il cui
    cord_uid non ha parse pmc. Ritorna un dask DataFrame filtrato."""
    pmc_uids = (paragraphs_ddf[paragraphs_ddf["source"] == "pmc"]
                ["cord_uid"].unique().compute())
    pmc_set = set(pmc_uids)
    keep = ((paragraphs_ddf["source"] == "pmc")
            | (~paragraphs_ddf["cord_uid"].isin(pmc_set)))
    return paragraphs_ddf[keep]


# le etichette di sezione sono testo libero e non canonicalizzabili (~1M distinte); ci
# limitiamo a FLAGGARE i paragrafi la cui sezione sembra boilerplate / non-corpo.
REFERENCE_SECTION_RE = (r"(?:referen|bibliograph|acknowledg|author contrib|"
                        r"conflict|competing interest|funding|declarat|"
                        r"supplement|copyright)")

PARA_SILVER_SCHEMA = pa.schema([("cord_uid", pa.string()), ("paper_id", pa.string()),
                                ("source", pa.string()), ("para_idx", pa.int32()),
                                ("section", pa.string()), ("text", pa.string()),
                                ("is_reference_like", pa.bool_())])

In [ ]:
# patch ISO3 per i valori che la regex di country_converter manca (nomi in lingua,
# abbreviazioni). Le chiavi sono accent-folded, minuscole, solo lettere.
COUNTRY_ALIASES = {
    "deutschland": "DEU", "espana": "ESP", "brasil": "BRA", "mexico": "MEX",
    "osterreich": "AUT", "uae": "ARE", "uk": "GBR", "scotland": "GBR",
    "england": "GBR", "wales": "GBR", "northern ireland": "GBR",
    "the netherlands": "NLD", "usa": "USA",
}


def accent_key(s):
    """Fold degli accenti, minuscolo, solo lettere/spazi -> chiave di alias stabile."""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z ]", " ", s.lower()).strip()


def first_valid_iso3(v, iso3_set):
    """coco ritorna: una str (1 match), una lista (multi -> primo valido), o l'input
    invariato (nessun match -> non in iso3_set -> None)."""
    if isinstance(v, list):
        return next((x for x in v if x in iso3_set), None)
    return v if v in iso3_set else None


def canonicalize_country(country_raw):
    """Series country_raw -> (Series iso3, Series name). Tre passate sui valori DISTINTI:
    (1) regex di country_converter, (2) patch di alias, (3) retry sul primo token."""
    import logging
    import country_converter as coco
    logging.getLogger("country_converter").setLevel(logging.CRITICAL)
    cc = coco.CountryConverter()
    iso3_set = set(cc.data.ISO3.dropna())
    iso3_name = dict(zip(cc.data.ISO3, cc.data.name_short))

    distinct = list(pd.Index(country_raw.dropna().unique()))
    conv = cc.convert(distinct, to="ISO3", not_found=None)     # passata 1 (vettoriale)
    mapping, todo = {}, []
    for raw, v in zip(distinct, conv):
        iso = first_valid_iso3(v, iso3_set)
        if iso:
            mapping[raw] = iso
        else:
            todo.append(raw)
    for raw in todo:
        iso = COUNTRY_ALIASES.get(accent_key(raw))             # passata 2 (alias)
        if iso is None:                                        # passata 3 (primo token)
            head = re.sub(r"[^A-Za-z ]", " ", re.split(r"[,;/]", raw)[0]).strip()
            if head:
                iso = first_valid_iso3(cc.convert(head, to="ISO3", not_found=None), iso3_set)
                if iso is None:
                    iso = COUNTRY_ALIASES.get(accent_key(head))
        mapping[raw] = iso
    iso3 = country_raw.map(mapping)
    return iso3, iso3.map(iso3_name)


def norm_institution(s):
    """Normalizzazione leggera (la disambiguazione completa è fuori scope): unicode-normalize,
    collassa gli spazi, strip della punteggiatura ai bordi."""
    if not isinstance(s, str):
        return None
    s = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s)).strip(" ,;.-\t")
    return s or None

## 9 · Costruzione linkage + work-item

In modalità **sample** (`SAMPLE > 0`) si campiona un sottoinsieme riproducibile di file per
sorgente (`random.seed(0)`) e si limitano le righe lette da `metadata.csv`, così il dry-run è
veloce e scrive in `data_sample/`. `unresolved == 0` verifica che ogni file su disco sia
riconducibile ad almeno un `cord_uid`.

In [ ]:
print("costruzione linkage + work-item ...")
pdf_map, pmc_map = build_linkage()
pdf_items, pmc_items, unresolved = build_workitems(pdf_map, pmc_map)
assert unresolved == 0, f"{unresolved} file non risolti"
print(f"  pdf={len(pdf_items)} pmc={len(pmc_items)} unresolved=0")

# dry-run: campiona un sottoinsieme riproducibile di file/sorgente e limita le righe di metadata
meta_nrows = None
if SAMPLE:
    random.seed(0)
    pdf_items = random.sample(pdf_items, min(SAMPLE, len(pdf_items)))
    pmc_items = random.sample(pmc_items, min(SAMPLE, len(pmc_items)))
    meta_nrows = SAMPLE * 3
print(f"  in uso: pdf={len(pdf_items)} pmc={len(pmc_items)} meta_nrows={meta_nrows}")

## 10 · Build BRONZE

`papers` è piccola (≤425k righe) → **pandas puro**, nessun task graph. `paragraphs` e `authors`
sono grandi → **Dask Bag**: `from_sequence(work-item) → map(estrai) → flatten → to_dataframe →
to_parquet` con schema Arrow esplicito e compressione `zstd`. `authors` usa solo i `pdf_items`
(le affiliazioni PMC sono vuote).

In [ ]:
# papers: piccola (<=425k righe) -> pandas puro, nessun task graph Dask
t0 = time.time()
papers_raw = pd.read_csv(META, dtype=str, nrows=meta_nrows)
bpath = os.path.join(ROOT, "bronze", "papers")
write_pandas_parquet(papers_raw.astype("string"), bpath, NPART_PAPERS)
report("bronze/papers", bpath, t0)

In [ ]:
# paragraphs: grande -> Dask Bag (map estrazione -> flatten -> DataFrame -> Parquet)
t0 = time.time()
items = pdf_items + pmc_items
bpath = os.path.join(ROOT, "bronze", "paragraphs")
fresh_dir(bpath)
bag = db.from_sequence(items, npartitions=NPART_PARA).map(extract_paragraphs).flatten()
bag.to_dataframe(meta=PARA_META).to_parquet(
    bpath, engine="pyarrow", write_index=False, compression="zstd", schema=PARA_SCHEMA)
report("bronze/paragraphs", bpath, t0)

In [ ]:
# authors: solo pdf_items (le affiliazioni in pmc_json sono ~0% popolate)
t0 = time.time()
bpath = os.path.join(ROOT, "bronze", "authors")
fresh_dir(bpath)
bag = db.from_sequence(pdf_items, npartitions=NPART_AUTH).map(extract_authors).flatten()
bag.to_dataframe(meta=AUTH_META).to_parquet(
    bpath, engine="pyarrow", write_index=False, compression="zstd", schema=AUTH_SCHEMA)
report("bronze/authors", bpath, t0)

## 11 · Build SILVER

Costruito **dal bronze** (nessun re-parse dei JSON). Rispetto ai vecchi script, il silver viene
scritto **una sola volta** nella versione arricchita: prima `build_parquet.py` ne scriveva una
versione base subito sovrascritta da `build_silver.py`. L'output finale `silver/` è identico, ma
senza doppia scrittura.

Oltre a `papers`, `authors` e `paragraphs`, si producono due **rollup** distinti per la task 2:
`paper_countries` (paper × paese) e `paper_institutions` (paper × istituto).

In [ ]:
# silver/papers: dedup + year + flag + drop colonne morte, poi flag sui titoli
t0 = time.time()
papers = pd.read_parquet(os.path.join(ROOT, "bronze", "papers"))
papers = silver_papers(papers)
papers = enrich_papers(papers)
spath = os.path.join(ROOT, "silver", "papers")
write_pandas_parquet(papers, spath, NPART_PAPERS)
report("silver/papers", spath, t0)

In [ ]:
# silver/authors: canonicalizzazione paese (ISO3 + nome) + normalizzazione istituto
t0 = time.time()
authors = pd.read_parquet(os.path.join(ROOT, "bronze", "authors"))
iso3, name = canonicalize_country(authors["country_raw"])
authors["country_iso3"] = iso3
authors["country"] = name
authors["institution_norm"] = authors["institution"].map(norm_institution)
authors = authors[["cord_uid", "paper_id", "author_idx", "institution", "institution_norm",
                   "country_raw", "country_iso3", "country", "settlement"]]
spath = os.path.join(ROOT, "silver", "authors")
write_pandas_parquet(authors, spath, NPART_AUTH)
resolved = int(authors["country_iso3"].notna().sum())
nonnull = int(authors["country_raw"].notna().sum())
print(f"       country risolti: {resolved}/{nonnull} "
      f"({100*resolved/max(nonnull,1):.2f}% dei raw non-null)")
report("silver/authors", spath, t0)

In [ ]:
# rollup: una riga per (paper, paese) e per (paper, istituto)
t0 = time.time()
paper_countries = (authors.loc[authors.country_iso3.notna(),
                               ["cord_uid", "country_iso3", "country"]]
                   .drop_duplicates().reset_index(drop=True))
opath = os.path.join(ROOT, "silver", "paper_countries")
write_pandas_parquet(paper_countries, opath, NPART_PAPERS)
report("silver/paper_countries", opath, t0)

t0 = time.time()
paper_institutions = (authors.loc[authors.institution_norm.notna(),
                                  ["cord_uid", "institution_norm"]]
                      .drop_duplicates().reset_index(drop=True))
opath = os.path.join(ROOT, "silver", "paper_institutions")
write_pandas_parquet(paper_institutions, opath, NPART_PAPERS)
report("silver/paper_institutions", opath, t0)

In [ ]:
# silver/paragraphs: prefer-pmc + flag is_reference_like (Dask, no re-parse dei JSON)
t0 = time.time()
para = dd.read_parquet(os.path.join(ROOT, "bronze", "paragraphs"), engine="pyarrow")
para = silver_paragraphs(para)
para = para.assign(is_reference_like=para["section"].fillna("").str.lower()
                   .str.contains(REFERENCE_SECTION_RE, regex=True))
spath = os.path.join(ROOT, "silver", "paragraphs")
fresh_dir(spath)
para.to_parquet(spath, engine="pyarrow", write_index=False,
                compression="zstd", schema=PARA_SILVER_SCHEMA)
report("silver/paragraphs", spath, t0)

## 12 · Sanity check

Controlli sui dataset prodotti: dimensioni, **unicità** di `cord_uid`, **integrità referenziale**
(`authors`/`paragraphs` ⊆ `papers`), invariante **prefer-pmc** (nessun paper tiene entrambe le
sorgenti), copertura del body text. Gli assert davvero **strutturali** (unicità di `cord_uid`,
invariante prefer-pmc) valgono sempre. Quelli che **confrontano tabelle diverse** (integrità
referenziale) o usano **conteggi assoluti** (es. 406 211 paper) valgono solo sul run completo:
in modalità sample `papers` è una *testa* di `metadata.csv` mentre i file sono campionati a caso,
quindi gli insiemi di `cord_uid` sono disgiunti.

In [ ]:
def dirstat(path):
    parts = [f for f in os.listdir(path) if f.endswith(".parquet")]
    size = sum(os.path.getsize(os.path.join(path, f)) for f in parts)
    return len(parts), size / 1e6


def banner(m):
    print("\n" + "=" * 68 + f"\n{m}\n" + "=" * 68)


banner("DATASET (part-file, dimensione)")
for name in ["bronze/papers", "silver/papers", "bronze/paragraphs", "silver/paragraphs",
             "bronze/authors", "silver/authors", "silver/paper_countries",
             "silver/paper_institutions"]:
    n, mb = dirstat(os.path.join(ROOT, name))
    print(f"  {name:26s} {n:3d} parts  {mb:9.1f} MB")

In [ ]:
banner("PAPERS (silver)")
ps = pd.read_parquet(os.path.join(ROOT, "silver", "papers"),
                     columns=["cord_uid", "title", "year", "has_pdf", "has_pmc"])
print("righe:", len(ps), "| cord_uid unico:", ps.cord_uid.is_unique)
assert ps.cord_uid.is_unique
if not SAMPLE:
    assert len(ps) == 406211, "attesi 406211 paper unici sul run completo"
print("anno picco:", int(ps.year.value_counts().idxmax()), "->", int(ps.year.value_counts().max()))
print("has_pdf:", int(ps.has_pdf.sum()), "| has_pmc:", int(ps.has_pmc.sum()),
      "| title null:", int(ps.title.isna().sum()))
papers_set = set(ps.cord_uid)

In [ ]:
banner("AUTHORS (bronze, task 2)")
au = pd.read_parquet(os.path.join(ROOT, "bronze", "authors"),
                     columns=["cord_uid", "institution", "country_raw"])
print("righe:", len(au), "| paper distinti:", au.cord_uid.nunique())
print("institution non-null: {} ({:.1f}%) | country non-null: {} ({:.1f}%)".format(
    int(au.institution.notna().sum()), 100 * au.institution.notna().mean(),
    int(au.country_raw.notna().sum()), 100 * au.country_raw.notna().mean()))
print("top 8 country_raw (sporchi):")
print(au.country_raw.value_counts().head(8).to_string())
auth_set = set(au.cord_uid.unique())
if SAMPLE:
    # sample: papers = testa di metadata, file campionati a caso -> insiemi disgiunti (atteso)
    print(f"(sample) authors.cord_uid non in papers: {len(auth_set - papers_set)} (atteso)")
else:
    assert auth_set <= papers_set, "authors referenzia un cord_uid non in papers!"
    print("integrita referenziale authors.cord_uid subset di papers.cord_uid: OK")

In [ ]:
banner("PARAGRAPHS (bronze vs silver, task 1)")
pb = dd.read_parquet(os.path.join(ROOT, "bronze", "paragraphs"), columns=["cord_uid", "source"])
sp = dd.read_parquet(os.path.join(ROOT, "silver", "paragraphs"), columns=["cord_uid", "source"])

pb_n = int(pb.shape[0].compute())
pb_src = pb.source.value_counts().compute().to_dict()
pb_uids = int(pb.cord_uid.nunique().compute())
print(f"bronze: righe={pb_n}  per sorgente={pb_src}  paper distinti={pb_uids}")

sp_n = int(sp.shape[0].compute())
sp_src = sp.source.value_counts().compute().to_dict()
print(f"silver: righe={sp_n}  per sorgente={sp_src}")
assert set(sp_src) <= {"pdf", "pmc"}, "valore di source inatteso"

pmc_uids = set(sp[sp.source == "pmc"].cord_uid.unique().compute())
pdf_uids = set(sp[sp.source == "pdf"].cord_uid.unique().compute())
overlap = pmc_uids & pdf_uids
print(f"silver paper distinti: pmc={len(pmc_uids)} solo-pdf={len(pdf_uids)} "
      f"totale={len(pmc_uids | pdf_uids)}")
assert len(overlap) == 0, f"prefer-pmc violato: {len(overlap)} paper tengono entrambe le sorgenti"
print("invariante prefer-pmc (nessun paper con entrambe le sorgenti): OK")

sil_para_uids = pmc_uids | pdf_uids
if SAMPLE:
    print(f"(sample) paragraphs.cord_uid non in papers: {len(sil_para_uids - papers_set)} (atteso)")
else:
    assert sil_para_uids <= papers_set, "paragraphs referenzia un cord_uid non in papers!"
    print("integrita referenziale paragraphs.cord_uid subset di papers.cord_uid: OK")

cov = int((ps.has_pdf | ps.has_pmc).sum())
print(f"copertura: {len(sil_para_uids)} paper con body text "
      f"(su {cov} paper con >=1 parse; il gap = parse con body vuoto)")

banner("TUTTI I SANITY CHECK PASSATI")

## 13 · Chiusura

Chiude client e (se locale) cluster. Rieseguendo il notebook dall'inizio, la cella idempotente
del §2 li richiude comunque.

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("cluster chiuso.")

## Appendice · Come eseguire sulla VM

Sulla VM **non si edita il notebook**: si impostano le env var nella shell (dentro `tmux`) e si
esegue su una **copia usa-e-getta**, così il file tracciato da git resta identico e `git pull`
non entra mai in conflitto.

```bash
export CORD19_ARCHIVE=/data/kagglehub/.../CORD-19-research-challenge/<...>
export CORD19_DATA=/data/parquet
# cluster multi-nodo via SSH (il notebook gira sullo scheduler):
export CORD19_HOSTS="scheduler,scheduler,worker1,worker2"    # 1° = scheduler; tutti = worker
export CORD19_SSH_KEY=~/.ssh/mapd-cluster.pem               # chiave privata unica del cluster
# (in alternativa, per uno scheduler già avviato: export DASK_SCHEDULER=tcp://scheduler:8786)
jupyter nbconvert --to notebook --execute \
  --ExecutePreprocessor.kernel_name=<il-tuo-kernel> \
  --output /tmp/executed_conversion.ipynb \
  conversion_sanification.ipynb
```

`CORD19_HOSTS` accetta hostname (se risolti in `/etc/hosts`) o IP interni `10.67.22.x`.
Gli output dei dati vanno su `CORD19_DATA` (già git-ignored). Per un **dry-run** veloce (cluster
locale sul singolo nodo): lascia vuoti `CORD19_HOSTS`/`DASK_SCHEDULER` e usa `CORD19_SAMPLE=2000`.